## Loading the lora adapter

In [ ]:
adapter_path="/content/drive/MyDrive/lfm-profanity-lora"

## Downloading the llm model from hugging face

Also attacing the pretrained lora adapter to the model

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model_name = "LiquidAI/LFM2.5-1.2B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(adapter_path, local_files_only=True)

print("Downloading/loading base model (this may take a few minutes on first run)...")
model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.float16,  # Changed from torch_dtype (deprecated)
    device_map="auto",
    trust_remote_code=True,
)

print("Loading LoRA adapters...")
model = PeftModel.from_pretrained(model, adapter_path, local_files_only=True)

print("Model loaded successfully!")





Loading tokenizer...
Downloading/loading base model (this may take a few minutes on first run)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Loading LoRA adapters...
Model loaded successfully!


## Creating the review analysis function:

This function here will:
- Detect profanity in the given review
- Detect the sentiment of the review provided
- If the given review contains any profanity, it will rewrite the review without the profanity


In [ ]:
def hybrid_analyze_review(review_text):
    # ====================================================
    # STEP 1: CLASSIFICATION (Adapter ON)
    # ====================================================
    detect_prompt = f"""<|im_start|>system
You are a helpful AI assistant specialized in content moderation.<|im_end|>
<|im_start|>user
Analyze the following customer review and provide:
1. Whether it contains profanity (yes/no)
2. The sentiment (positive/negative/neutral)

Review: {review_text}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer([detect_prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.1)
    analysis = (
        tokenizer.decode(outputs[0], skip_special_tokens=True)
        .split("assistant")[-1]
        .strip()
    )

    # Parse logic
    has_profanity = "Profanity: yes" in analysis
    is_positive = "Sentiment: positive" in analysis.lower()

    # ====================================================
    # STEP 2: REWRITE (Base Model - Adapter OFF)
    # ====================================================
    if has_profanity:
        if is_positive:
            # POSITIVE CASE
            system_task = (
                "You are a text rewriting engine. "
                "Your task is to remove profanity from enthusiastic praise while preserving excitement."
            )

            constraints = (
                "Transformation rules:\n"
                "- Preserve the original enthusiasm and positive emphasis.\n"
                "- Replace profanity with strong but professional praise.\n"
                "- Do NOT weaken the sentiment.\n"
                "- Do NOT summarize or neutralize excitement.\n"
                "- Output only the rewritten review text."
            )

            examples = """
          Original: Holy shit, this espresso machine is amazing!
          Rewritten: This espresso machine is absolutely amazing!

          Original: This game is fucking incredible. I can't put it down.
          Rewritten: This game is incredibly engaging. I cannot put it down.

          Original: Hell yeah! Arrived in one day. You guys are kickass.
          Rewritten: Excellent! It arrived in one day and the service was outstanding.
          """
            temperature = 0.6

        else:
            # NEGATIVE CASE
            system_task = (
                "You are a text rewriting engine. "
                "Your task is to remove profanity and insults while preserving the original complaint exactly."
            )

            constraints = (
                "Transformation rules:\n"
                "- Perform minimal rewriting. Replace only the profane or insulting words.\n"
                "- Preserve all concrete details such as actions, timelines, failures, and outcomes.\n"
                "- Do NOT summarize, generalize, or abstract the complaint.\n"
                "- Do NOT remove mentions of money, delays, damage, or lack of response.\n"
                "- Maintain a professional but firm tone.\n"
                "- Output only the rewritten review text."
            )

            examples = """
          Original: This laptop is a piece of shit. It broke after two days. What the hell?
          Rewritten: This laptop is unacceptable. It broke after two days.

          Original: Don't buy from this bastard seller. They took my money and ghosted me.
          Rewritten: Do not buy from this seller. They took my money and stopped responding.

          Original: The food tasted like crap and the waiter was a total ass.
          Rewritten: The food tasted poor and the waiter was unprofessional.
          """
            temperature = 0.3

        # -------- PROMPT ASSEMBLY --------
        rewrite_prompt = f"""<|im_start|>system
    {system_task}

    {constraints}
    <|im_end|>
    <|im_start|>user
    Below are examples of correct rewrites:

    {examples}

    Rewrite the following review.

    Original:
    {review_text}

    Rewritten:
    <|im_end|>
    <|im_start|>assistant
    """

        inputs = tokenizer([rewrite_prompt], return_tensors="pt").to("cuda")

        with model.disable_adapter():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                temperature=temperature,
                do_sample=True,
            )

        rewrite = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # -------- SAFE CLEANUP --------
        rewrite = rewrite.split("assistant")[-1].strip()
        rewrite = rewrite.replace("Rewritten:", "").strip()
        rewrite = rewrite.splitlines()[0]

        return f"{analysis}\nRewritten: {rewrite}"

    else:
        return f"{analysis}\nRewritten: Not needed"


Testing the model

In [ ]:
test_cases = [
    # ---------------------------------------------------------
    # 1. PROFANE & NEGATIVE (The Standard Use Case)
    # Goal: Detect yes/negative, Rewrite to be polite.
    # ---------------------------------------------------------
    "This laptop is a piece of shit. It broke after two days. What the hell?",
    "Don't buy from this bastard seller. They took my money and ghosted me.",
    "The food tasted like crap and the waiter was a total ass.",

    # ---------------------------------------------------------
    # 2. NON-PROFANE & NEGATIVE (The "False Positive" Test)
    # Goal: Detect no/negative. Should NOT trigger a rewrite.
    # ---------------------------------------------------------
    "I am extremely disappointed. The item arrived damaged and the support team was unhelpful.",
    "Terrible experience. I waited an hour and the food was cold. Waste of money.",
    "Do not recommend. The quality is very poor and it feels cheap.",

    # ---------------------------------------------------------
    # 3. PROFANE & POSITIVE (The "Slang" Test)
    # Goal: Detect yes/positive. Rewrite should keep the COMPLIMENT.
    # ---------------------------------------------------------
    "Holy shit, this espresso machine is amazing! Best damn coffee I've ever had.",
    "This game is fucking incredible. I can't put it down.",
    "Hell yeah! Arrived in one day. You guys are kickass.",

    # ---------------------------------------------------------
    # 4. NON-PROFANE & POSITIVE (The Baseline)
    # Goal: Detect no/positive. Should NOT trigger a rewrite.
    # ---------------------------------------------------------
    "Absolutely lovely product. It works exactly as described.",
    "The customer service was outstanding. They resolved my issue in minutes.",
    "Five stars! I bought this for my mom and she loves it."
]

# ====================================================
# RUN THE BATCH TEST
# ====================================================
print(f" Running Hybrid Analysis on {len(test_cases)} examples...\n")

for i, review in enumerate(test_cases):
    print(f" Review #{i+1}: {review}")
    result = hybrid_analyze_review(review)
    print(result)
    print("-" * 60)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 Running Hybrid Analysis on 12 examples...

 Review #1: This laptop is a piece of shit. It broke after two days. What the hell?
Profanity: yes
Sentiment: negative
Rewritten: This laptop is not satisfactory. It malfunctioned after only two days of use.
------------------------------------------------------------
 Review #2: Don't buy from this bastard seller. They took my money and ghosted me.
Profanity: yes
Sentiment: negative
Rewritten: Do not purchase from this seller. They failed to deliver and did not respond.
------------------------------------------------------------
 Review #3: The food tasted like crap and the waiter was a total ass.
Profanity: yes
Sentiment: negative
Rewritten: The food did not meet expectations, and the service was disappointing.
------------------------------------------------------------
 Review #4: I am extremely disappointed. The item arrived damaged and the support team was unhelpful.
Profanity: no
Sentiment: negative
Rewritten: Not needed
-------------

## With adapter on:

In [ ]:
def hybrid_analyze_review_adapter_on(review_text):
    # ====================================================
    # STEP 1: CLASSIFICATION (Adapter ON)
    # ====================================================
    detect_prompt = f"""<|im_start|>system
You are a helpful AI assistant specialized in content moderation.<|im_end|>
<|im_start|>user
Analyze the following customer review and provide:
1. Whether it contains profanity (yes/no)
2. The sentiment (positive/negative/neutral)

Review: {review_text}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer([detect_prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.1)
    analysis = (
        tokenizer.decode(outputs[0], skip_special_tokens=True)
        .split("assistant")[-1]
        .strip()
    )

    # Parse logic
    has_profanity = "Profanity: yes" in analysis
    is_positive = "Sentiment: positive" in analysis.lower()

    # ====================================================
    # STEP 2: REWRITE (Base Model - Adapter ON - as per user request)
    # ====================================================
    if has_profanity:
        if is_positive:
            # POSITIVE CASE
            system_task = (
                "You are a text rewriting engine. "
                "Your task is to remove profanity from enthusiastic praise while preserving excitement."
            )

            constraints = (
                "Transformation rules:\n"
                "- Preserve the original enthusiasm and positive emphasis.\n"
                "- Replace profanity with strong but professional praise.\n"
                "- Do NOT weaken the sentiment.\n"
                "- Do NOT summarize or neutralize excitement.\n"
                "- Output only the rewritten review text."
            )

            examples = """
          Original: Holy shit, this espresso machine is amazing!
          Rewritten: This espresso machine is absolutely amazing!

          Original: This game is fucking incredible. I can't put it down.
          Rewritten: This game is incredibly engaging. I cannot put it down.

          Original: Hell yeah! Arrived in one day. You guys are kickass.
          Rewritten: Excellent! It arrived in one day and the service was outstanding.
          """
            temperature = 0.6

        else:
            # NEGATIVE CASE
            system_task = (
                "You are a text rewriting engine. "
                "Your task is to remove profanity and insults while preserving the original complaint exactly."
            )

            constraints = (
                "Transformation rules:\n"
                "- Perform minimal rewriting. Replace only the profane or insulting words.\n"
                "- Preserve all concrete details such as actions, timelines, failures, and outcomes.\n"
                "- Do NOT summarize, generalize, or abstract the complaint.\n"
                "- Do NOT remove mentions of money, delays, damage, or lack of response.\n"
                "- Maintain a professional but firm tone.\n"
                "- Output only the rewritten review text."
            )

            examples = """
          Original: This laptop is a piece of shit. It broke after two days. What the hell?
          Rewritten: This laptop is unacceptable. It broke after two days.

          Original: Don't buy from this bastard seller. They took my money and ghosted me.
          Rewritten: Do not buy from this seller. They took my money and stopped responding.

          Original: The food tasted like crap and the waiter was a total ass.
          Rewritten: The food tasted poor and the waiter was unprofessional.
          """
            temperature = 0.3

        # -------- PROMPT ASSEMBLY --------
        rewrite_prompt = f"""<|im_start|>system
    {system_task}

    {constraints}
    <|im_end|>
    <|im_start|>user
    Below are examples of correct rewrites:

    {examples}

    Rewrite the following review.

    Original:
    {review_text}

    Rewritten:
    <|im_end|>
    <|im_start|>assistant
    """

        inputs = tokenizer([rewrite_prompt], return_tensors="pt").to("cuda")

        # Adapter will be ON here as requested by the user
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=temperature,
            do_sample=True,
        )

        rewrite = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # -------- SAFE CLEANUP --------
        rewrite = rewrite.split("assistant")[-1].strip()
        rewrite = rewrite.replace("Rewritten:", "").strip()

        # Handle cases where rewrite might be empty after cleanup
        cleaned_rewrite_lines = rewrite.splitlines()
        if cleaned_rewrite_lines:
            rewrite = cleaned_rewrite_lines[0]
        else:
            rewrite = "[Rewrite generation failed or was empty]"

        return f"{analysis}\nRewritten: {rewrite}"

    else:
        return f"{analysis}\nRewritten: Not needed"

In [ ]:
test_cases = [
    # ---------------------------------------------------------
    # 1. PROFANE & NEGATIVE (The Standard Use Case)
    # Goal: Detect yes/negative, Rewrite to be polite.
    # ---------------------------------------------------------
    "This laptop is a piece of shit. It broke after two days. What the hell?",
    "Don't buy from this bastard seller. They took my money and ghosted me.",
    "The food tasted like crap and the waiter was a total ass.",

    # ---------------------------------------------------------
    # 2. NON-PROFANE & NEGATIVE (The "False Positive" Test)
    # Goal: Detect no/negative. Should NOT trigger a rewrite.
    # ---------------------------------------------------------
    "I am extremely disappointed. The item arrived damaged and the support team was unhelpful.",
    "Terrible experience. I waited an hour and the food was cold. Waste of money.",
    "Do not recommend. The quality is very poor and it feels cheap.",

    # ---------------------------------------------------------
    # 3. PROFANE & POSITIVE (The "Slang" Test)
    # Goal: Detect yes/positive. Rewrite should keep the COMPLIMENT.
    # ---------------------------------------------------------
    "Holy shit, this espresso machine is amazing! Best damn coffee I've ever had.",
    "This game is fucking incredible. I can't put it down.",
    "Hell yeah! Arrived in one day. You guys are kickass.",

    # ---------------------------------------------------------
    # 4. NON-PROFANE & POSITIVE (The Baseline)
    # Goal: Detect no/positive. Should NOT trigger a rewrite.
    # ---------------------------------------------------------
    "Absolutely lovely product. It works exactly as described.",
    "The customer service was outstanding. They resolved my issue in minutes.",
    "Five stars! I bought this for my mom and she loves it."
]

# ====================================================
# RUN THE BATCH TEST
# ====================================================
print(f" Running Hybrid Analysis on {len(test_cases)} examples...\n")

for i, review in enumerate(test_cases):
    print(f" Review #{i+1}: {review}")
    result = hybrid_analyze_review_adapter_on(review)
    print(result)
    print("-" * 60)

 Running Hybrid Analysis on 12 examples...

 Review #1: This laptop is a piece of shit. It broke after two days. What the hell?
Profanity: yes
Sentiment: negative
Rewritten: [Rewrite generation failed or was empty]
------------------------------------------------------------
 Review #2: Don't buy from this bastard seller. They took my money and ghosted me.
Profanity: yes
Sentiment: negative
Rewritten: [Rewrite generation failed or was empty]
------------------------------------------------------------
 Review #3: The food tasted like crap and the waiter was a total ass.
Profanity: yes
Sentiment: negative
Rewritten: The food tasted poor and the waiter was unprofessional.
------------------------------------------------------------
 Review #4: I am extremely disappointed. The item arrived damaged and the support team was unhelpful.
Profanity: no
Sentiment: negative
Rewritten: Not needed
------------------------------------------------------------
 Review #5: Terrible experience. I waited

## Generating reviews:
this function here is using the model to genrate random reviews about a random product

In [ ]:
import random

def generate_random_review(sentiment_type, product = None):
    """
    Generates a random review using the Base Model (Adapter OFF).
    """

    # 1. Randomize the Product to keep it interesting
    products = [
        "a high-end gaming laptop", "a local pizza place", "a pair of running shoes",
        "a new smartphone app", "a vacuum cleaner", "customer support service",
        "an espresso machine", "a violent video game"
    ]
    if not product:
      product = random.choice(products)

    # 2. Build Prompt based on User Request
    if "positive" in sentiment_type.lower():
        # We ask for "slang" so we can test if your Analyzer catches 'kickass' vs 'lovely'
        instruction = (
            f"Write a short, enthusiastic 5-star review for {product}. "
            "Use casual slang (like 'kickass', 'beast', 'insane') to sound authentic."
        )
    elif "negative" in sentiment_type.lower():
        # We ask for "mild profanity" so we can test if your Analyzer cleans it up
        instruction = (
            f"Write a short, angry 1-star review for {product}. "
            "You are frustrated. Use mild profanity (like 'crap', 'hell', 'garbage') to vent."
        )
    else:
        return "Error: Please choose 'positive' or 'negative'."

    # 3. Format for LFM-2.5
    prompt = f"""<|im_start|>system
You are a customer writing a review on a website. Be brief and expressive.<|im_end|>
<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
"""

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # 4. GENERATE (Adapter OFF)
    # We use High Temperature (0.9) to make it different every time
    with model.disable_adapter():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.9,
            do_sample=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip()

An interactive demo to show the capabilities of the mode

In [ ]:
def run_interactive_demo():
    print("--- AI REVIEW GENERATOR & FIXER ---")
    print("Type 'exit' to quit.\n")
    while True:

        user_choice = input("\nWhat kind of review do you want? (positive/negative): ").strip().lower()

        if user_choice == 'exit':
            print("Goodbye!")
            break

        product = input("\nWhat do you want the review about?: ").strip().lower()

        if user_choice not in ['positive', 'negative']:
            print("Please type 'positive' or 'negative'.")
            continue

        # 2. Generate the Random Review
        print(f"\n...Dreaming up a {user_choice} review for {product}...")
        generated_review = generate_random_review(user_choice, product)

        print("-" * 50)
        print(f" GENERATED REVIEW:\n\"{generated_review}\"")
        print("-" * 50)

        # 3. Analyze and Fix it automatically
        print("...Running Hybrid Analysis Pipeline...")
        result = hybrid_analyze_review(generated_review)

        print("\n ANALYSIS RESULT:")
        print(result)
        print("=" * 50)

# START THE APP
run_interactive_demo()

--- AI REVIEW GENERATOR & FIXER ---
Type 'exit' to quit.



KeyboardInterrupt: Interrupted by user